# Episodic Memory with Write-Time Reflection

This notebook demonstrates `ReflectiveMemory` and `JSONMemoryStore` working together
to give an `LLMAgent` episodic memory that distils a short lesson from each completed
task at write time.

The example is drawn directly from the
[Reflexion paper (Shinn et al., 2023)](https://arxiv.org/abs/2303.11366), which
frames the pattern as verbal reinforcement learning: an evaluator judges whether the
agent's output is correct, a self-reflection model converts that judgement into a
natural-language lesson, and episodic memory stores the lesson for future retrieval.

The task here is Monte Carlo estimation of pi. The target range `[3.1415, 3.1425)`
acts as the built-in **evaluator** — the agent checks its own estimate against the
range and knows immediately whether it succeeded or failed. That explicit pass/fail
signal is what makes the reflection genuinely informative: when the agent misses the
range it can articulate *why*, and that lesson is directly applicable to the next run.

Two properties are illustrated:

- **Write-time reflection.** After each task `ReflectiveMemory` calls the LLM to
  produce a one-sentence lesson from the task and its outcome. That lesson is attached
  to the episode under `additional_data["reflection"]` and rendered automatically
  inside a `<reflection>` tag whenever the episode is displayed.
- **Recalled lessons.** When a new task arrives, `recall()` surfaces the three most
  recent episodes. Because each episode carries a pre-distilled lesson, the agent can
  apply the insight on the next attempt rather than rediscovering the same mistakes.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code in this notebook you need Ollama installed locally with its
service running. Download Ollama from https://ollama.com/download, then start the
service by running `ollama serve` in a terminal.

## Setup

In [ ]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"✓ Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"✓ Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


ensure_ollama()

In [ ]:
import logging
from pathlib import Path

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.data_structures.memory import Episode
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging
from llm_agents_from_scratch.memory import JSONMemoryStore, ReflectiveMemory

## Part 1 — Tools for Monte Carlo Pi Estimation

Three tools implement the Monte Carlo algorithm. `generate_random_sample` seeds a
new sample in a global registry, `add_more_points_to_sample` grows that sample
without discarding existing data, and `monte_carlo_estimate` computes the current
pi estimate from any stored sample.

The task instruction tells the agent what the tools do but prescribes no algorithm.
The agent must decide how many points to start with, when to add more, and —
critically — whether to call `add_more_points_to_sample` or start fresh with
`generate_random_sample` when refining its estimate. Creating a new sample throws
away all accumulated data; adding to the existing one converges faster. That
mistake, when it happens, generates the most useful reflections.

In [ ]:
import uuid

import numpy as np
from pydantic import BaseModel, ConfigDict, Field, computed_field

from llm_agents_from_scratch.tools import PydanticFunctionTool

SAMPLE_REGISTRY: dict[str, list[tuple[float, float]]] = {}


class RandomSampleParams(BaseModel):
    """Params for generate_random_sample."""

    model_config = ConfigDict(extra="forbid")
    n: int = Field(description="Number of random points to generate.")


class RandomSample(BaseModel):
    """Result from generate_random_sample or add_more_points_to_sample."""

    sample_id: str = Field(
        description="Pass this sample_id to monte_carlo_estimate.",
    )

    @computed_field
    @property
    def sample_size(self) -> int:
        """Current number of points in the sample."""
        return len(SAMPLE_REGISTRY[self.sample_id])

    def __str__(self) -> str:
        """JSON string representation."""
        return self.model_dump_json()


def generate_random_sample(params: RandomSampleParams) -> RandomSample:
    """Generate n random points in [0,1]x[0,1] and store them.

    Returns a sample_id. Pass this sample_id to monte_carlo_estimate.
    """
    pts = np.random.uniform(size=(params.n, 2))
    sample_id = str(uuid.uuid4())
    SAMPLE_REGISTRY[sample_id] = [tuple(pt) for pt in pts.tolist()]
    return RandomSample(sample_id=sample_id)


class AddPointsParams(BaseModel):
    """Params for add_more_points_to_sample."""

    model_config = ConfigDict(extra="forbid")
    sample_id: str = Field(
        description="The sample_id of the sample to grow.",
    )
    n: int = Field(description="Number of additional random points to add.")


def add_more_points_to_sample(params: AddPointsParams) -> RandomSample:
    """Append n more random points to an existing sample.

    Returns the same sample_id with an updated sample size.
    """
    pts = np.random.uniform(size=(params.n, 2))
    SAMPLE_REGISTRY[params.sample_id] += [tuple(pt) for pt in pts.tolist()]
    return RandomSample(sample_id=params.sample_id)


class MonteCarloParams(BaseModel):
    """Params for monte_carlo_estimate."""

    model_config = ConfigDict(extra="forbid")
    sample_id: str = Field(
        description="The sample_id returned by generate_random_sample.",
    )


class MonteCarloResult(BaseModel):
    """Result from monte_carlo_estimate."""

    sample_id: str
    sample_size: int
    estimate: float

    def __str__(self) -> str:
        """JSON string representation."""
        return self.model_dump_json()


def monte_carlo_estimate(params: MonteCarloParams) -> MonteCarloResult:
    """Estimate pi using the Monte Carlo method.

    Counts the fraction of stored points that fall inside the unit circle
    and multiplies by 4.
    """
    points = SAMPLE_REGISTRY[params.sample_id]
    n = len(points)
    inside = sum(x**2 + y**2 < 1 for x, y in points)
    return MonteCarloResult(
        sample_id=params.sample_id,
        sample_size=n,
        estimate=(inside / n) * 4,
    )


random_sample_tool = PydanticFunctionTool(generate_random_sample)
add_points_tool = PydanticFunctionTool(add_more_points_to_sample)
mc_estimate_tool = PydanticFunctionTool(monte_carlo_estimate)

## Creating the Agent with Memory

`JSONMemoryStore` persists completed episodes to disk. `ReflectiveMemory` wraps the
store: before each episode is written it calls `llm.complete()` to distil a
one-sentence lesson from the task and result, then stores that lesson under
`episode.additional_data["reflection"]`.

Setting `n=1` means the most recent episode — with its embedded reflection — is
injected into the agent's system prompt when a new task begins.

In [ ]:
STORE_DIR = Path(".")
(STORE_DIR / "reflective_episodes.jsonl").unlink(missing_ok=True)

store = JSONMemoryStore(dir=STORE_DIR, filename="reflective_episodes.jsonl")
llm = OllamaLLM(model="qwen3:14b", think=False)
memory = ReflectiveMemory(store=store, llm=llm, n=1)
agent = LLMAgent(
    llm=llm,
    tools=[random_sample_tool, add_points_tool, mc_estimate_tool],
    memories=[memory],
)

## Part 2 — Running the Pi Estimation Task

The task has two requirements: reach an estimate in `[3.1415, 3.1425)` **and** do it
in at most 5 tool calls. The instruction tells the agent about the budget explicitly,
so the agent must plan its tool use — not just iterate until it happens to land in
range.

Five calls is tight. The naive approach — start with a small sample, call
`monte_carlo_estimate` after every small addition — will exhaust the budget without
converging. The winning strategy is to open with a large enough initial sample that
a single estimate already has a good chance of landing in range, leaving the remaining
calls for targeted additions if needed.

Whether the run succeeds or fails, `ReflectiveMemory` distils the outcome into a
one-sentence lesson grounded in the explicit pass/fail signal of the budget-and-range
constraint.

In [ ]:
TASK_INSTRUCTION = (
    "Estimate pi using Monte Carlo methods. "
    "Target: an estimate in the range [3.1415, 3.1425). "
    "Constraint: you have at most 5 tool calls total.\n\n"
    "Examples: 3.14159 is a success | "
    "3.14149 is too low | "
    "3.14250 is too high.\n\n"
    "You have three tools:\n"
    "- generate_random_sample(n): generates n random 2-D points in "
    "[0,1]x[0,1], stores them, and returns a sample_id.\n"
    "- add_more_points_to_sample(sample_id, n): appends n more points "
    "to the sample identified by sample_id.\n"
    "- monte_carlo_estimate(sample_id): computes a pi estimate from the "
    "points stored under sample_id.\n\n"
    "When your estimate is within the target range, respond with exactly:\n"
    '{"sample_id": "<the-sample-id>"}'
)

In [ ]:
enable_console_logging(logging.INFO)

task1 = Task(instruction=TASK_INSTRUCTION)
result1 = await agent.run(task1, max_steps=10)
print(result1)

In [ ]:
print(await memory.summary())

## The RecencyMemory Baseline

Before looking at the reflection, consider what `RecencyMemory` would surface for a
new task. A `RecencyMemory` stores no additional data. The recalled episode carries
the full result — which tools were called, what estimates were returned, whether the
target was reached — but the lesson is implicit.

A future agent seeing this raw episode would need to re-read the entire trajectory to
extract what strategy worked or failed. The signal is there, but buried.

In [ ]:
print("What RecencyMemory would recall (no reflection):\n")
recent_eps = await memory.store.read_recent(memory.n)
for ep in recent_eps:
    ep_without_reflection = Episode(
        task=ep.task,
        rollout=ep.rollout,
        result=ep.result,
        completed_at=ep.completed_at,
    )
    print(str(ep_without_reflection))
    print()

## Part 3 — Inspecting the Reflection

Now look at the same episode as stored by `ReflectiveMemory`. It carries a
`<reflection>` tag with the one-sentence lesson distilled at write time.

Because the Pi task has an explicit success criterion, the reflection model can
evaluate whether the strategy worked and articulate precisely what to do differently
next time — or confirm what to repeat if it succeeded. The lesson is captured in a
single sentence rather than buried in the full trajectory.

In [ ]:
print("What ReflectiveMemory recalls (with reflection):\n")
recent_eps = await memory.store.read_recent(memory.n)
for ep in recent_eps:
    print(str(ep))
    print()

## Part 4 — A Recalled Lesson Guides the Next Run

The same task is submitted a second time, this time without a step cap. Before the
agent takes its first step, `recall()` injects the episode from Part 2 — with its
embedded reflection — into the system prompt.

If Part 2 failed, the reflection names exactly what went wrong. If Part 2 succeeded,
the reflection confirms the winning strategy. Either way the agent starts this run
with a concrete lesson rather than a blank slate.

Check the logs: does the agent apply the recalled insight on its first tool call?

In [ ]:
print("Episode recalled for the second task:\n")
recalled = await memory.store.read_recent(memory.n)
for ep in recalled:
    print(str(ep))
    print()

In [ ]:
enable_console_logging(logging.INFO)

task4 = Task(instruction=TASK_INSTRUCTION)
result4 = await agent.run(task4)
print(result4)

## Key Takeaway

`ReflectiveMemory` converts task experience into transferable lessons. The Reflexion
paper frames this as verbal reinforcement learning: rather than updating model
weights, the agent stores natural-language feedback from each trial and retrieves it
when facing the same problem.

The Pi estimation task makes the mechanism concrete because it has a built-in
evaluator — the target range `[3.1415, 3.1425)`. When the agent misses the range it
knows it failed, and the reflection captures a corrective lesson (wrong tool choice,
too few points, stale estimate). When it succeeds the reflection confirms the
strategy. Either way, the next run starts with that knowledge.

The comparison in Parts 2 and 3 shows the difference in practice. Raw `RecencyMemory`
episodes surface the full trajectory — tool calls, estimates, corrections — but the
lesson is implicit. `ReflectiveMemory` surfaces it in a single sentence, so a future
agent can act on it without re-reading the entire history.

The separation of concerns matches `RecencyMemory` and `SimilarityMemory`:
`ReflectiveMemory` decides what to generate and what to attach; `JSONMemoryStore`
decides how to persist and retrieve. Swapping the store does not affect the reflection
logic, and any `BaseMemoryStore` implementing `read_recent()` works as a drop-in
backend.